> **Notebook d'approfondissement** — mise en forme visuelle des DataFrames pour rapports, présentations, exports Excel.

# Styling de DataFrames

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
from _data import load_accidents, load_penguins

df = load_accidents()
penguins = load_penguins()

## 1. Le principe : `df.style`

`df.style` retourne un objet `Styler` — il ne modifie pas les données, il ne modifie que leur rendu HTML.
Toutes les méthodes de styling se chaînent sur cet objet.

In [ ]:
# Table de référence : accidents par département, top 15
resume = (
    df
    .groupby("dep")
    .agg(
        nb_accidents=("Num_Acc", "count"),
        mois_le_plus_accidente=("mois", lambda df_: df_.value_counts().idxmax()),
        vma_mediane=("vma", "median"),
    )
    .nlargest(15, "nb_accidents")
    .reset_index()
)
resume

## 2. `format()` — contrôler l'affichage des valeurs

In [ ]:
# Formatage par colonne
resume.style.format({
    "nb_accidents":           "{:,}",       # séparateur de milliers
    "vma_mediane":            "{:.0f} km/h",
    "mois_le_plus_accidente": "Mois {}",
})

## 3. `highlight_max()`, `highlight_min()`, `highlight_null()`

In [ ]:
(
    resume
    .style
    .format({"nb_accidents": "{:,}", "vma_mediane": "{:.0f}"})
    .highlight_max(subset=["nb_accidents"], color="#ff9999")
    .highlight_min(subset=["nb_accidents"], color="#99ff99")
)

In [ ]:
# highlight_null : repérer les NaN dans le tableau
resume_avec_nan = resume.copy()
resume_avec_nan.loc[2, "vma_mediane"] = np.nan

resume_avec_nan.style.highlight_null(color="orange")

## 4. `background_gradient()` — dégradé de couleur

In [ ]:
# Dégradé sur une seule colonne
resume.style.background_gradient(subset=["nb_accidents"], cmap="Reds")

In [ ]:
# Dégradé sur toutes les colonnes numériques — attention : compare entre colonnes
resume.style.background_gradient(cmap="coolwarm")

In [ ]:
# Dégradé par colonne (axis=0) — chaque colonne a son propre range de couleurs
resume.style.background_gradient(cmap="Blues", axis=0)

## 5. `bar()` — barres de progression intégrées dans les cellules

In [ ]:
resume.style.bar(subset=["nb_accidents"], color="steelblue")

In [ ]:
# Barres centrées sur 0 pour les valeurs positives et négatives
# Ici on illustre avec des écarts à la médiane
med = resume["nb_accidents"].median()
resume_ecart = resume.assign(ecart=lambda df_: df_["nb_accidents"] - med)

resume_ecart.style.bar(
    subset=["ecart"],
    align="mid",
    color=["#FF4444", "#4CAF50"],  # rouge pour négatif, vert pour positif
)

## 6. `apply()` — style conditionnel personnalisé

Pour des règles qu'aucune méthode prête à l'emploi ne couvre,
`Styler.apply()` accepte une fonction qui retourne des strings CSS.

In [ ]:
def surligner_extremes(serie, bas=0.25, haut=0.75):
    """Rouge sur le quartile haut, vert sur le quartile bas."""
    q_bas  = serie.quantile(bas)
    q_haut = serie.quantile(haut)
    return [
        "background-color: #ff9999" if v > q_haut
        else "background-color: #99ff99" if v < q_bas
        else ""
        for v in serie
    ]

resume.style.apply(surligner_extremes, subset=["nb_accidents"])

In [ ]:
# Styler une ligne entière selon une condition
def surligner_paris(df_):
    """Fond jaune sur la ligne Paris (dep == '75')."""
    style = pd.DataFrame("", index=df_.index, columns=df_.columns)
    mask  = df_["dep"] == "75"
    style[mask] = "background-color: #fffacc"
    return style

resume.style.apply(surligner_paris, axis=None)

## 7. `set_properties()` — CSS global

In [ ]:
(
    resume
    .style
    .set_properties(**{"text-align": "center", "font-size": "12pt"})
    .set_properties(subset=["dep"], **{"font-weight": "bold"})
    .format({"nb_accidents": "{:,}"})
)

## 8. `set_caption()`, `hide()` — finitions

In [ ]:
(
    resume
    .style
    .set_caption("Top 15 des départements les plus accidentogènes — France 2024")
    .hide(axis="index")                    # masquer l'index (0, 1, 2…)
    .format({"nb_accidents": "{:,}", "vma_mediane": "{:.0f}"})
    .background_gradient(subset=["nb_accidents"], cmap="Oranges")
)

## 9. Chaînage complet — un rapport en une expression

In [ ]:
(
    resume
    .style
    .set_caption("Top 15 des départements — accidents France 2024")
    .hide(axis="index")
    .format({
        "nb_accidents":           "{:,}",
        "vma_mediane":            "{:.0f} km/h",
        "mois_le_plus_accidente": "Mois {:0.0f}",
    })
    .background_gradient(subset=["nb_accidents"], cmap="YlOrRd")
    .bar(subset=["vma_mediane"], color="steelblue")
    .highlight_max(subset=["nb_accidents"], color="#d73027")
    .set_properties(**{"text-align": "center"})
    .set_properties(subset=["dep"], **{"font-weight": "bold", "text-align": "left"})
)

## 10. Export

Le `Styler` peut s'exporter en HTML ou en Excel (avec `openpyxl`).

In [ ]:
styler = (
    resume
    .style
    .format({"nb_accidents": "{:,}", "vma_mediane": "{:.0f}"})
    .background_gradient(subset=["nb_accidents"], cmap="YlOrRd")
)

# Export HTML
with open("rapport_accidents.html", "w") as f:
    f.write(styler.to_html())
print("HTML écrit.")

In [ ]:
# Export Excel avec mise en forme (nécessite openpyxl)
try:
    styler.to_excel("rapport_accidents.xlsx", engine="openpyxl", index=False)
    print("Excel écrit avec mise en forme.")
except ImportError:
    print("openpyxl non installé : uv add openpyxl")

In [ ]:
import os
for f in ["rapport_accidents.html", "rapport_accidents.xlsx"]:
    if os.path.exists(f):
        os.remove(f)

## Récapitulatif

| Besoin | Méthode |
|---|---|
| Formater les nombres/dates | `.format({"col": "pattern"})` |
| Surligner min/max | `.highlight_min()` / `.highlight_max()` |
| Surligner les NaN | `.highlight_null(color=...)` |
| Dégradé de couleur | `.background_gradient(cmap=..., subset=...)` |
| Barres dans les cellules | `.bar(subset=..., color=...)` |
| Style conditionnel custom | `.apply(fonction, subset=...)` |
| CSS global | `.set_properties(**{"css-prop": "valeur"})` |
| Masquer l'index | `.hide(axis="index")` |
| Titre du tableau | `.set_caption("titre")` |
| Export HTML | `.to_html()` |
| Export Excel avec style | `.to_excel(engine="openpyxl")` |